# Fun & Fit 건강 어드바이저 에이전트 튜토리얼

이 노트북에서는 **Microsoft Foundry** SDK를 사용해 건강/피트니스 도메인의 기본 에이전트를 만들어봅니다.

진행 순서:
1. `azure-ai-projects`로 프로젝트 클라이언트를 초기화합니다.
2. 건강/영양/운동 관련 안내를 제공하는 에이전트를 생성합니다.
3. 스레드를 만들고 사용자 질문을 전달해 응답을 확인합니다.
4. 실행 로그를 확인하고 정리(cleanup)까지 수행합니다.

## 의료 안내 문구
> 이 노트북의 내용은 일반적인 교육 목적이며 전문적인 의학적 진단/치료를 대체하지 않습니다.

> 건강 상태와 관련된 의사결정은 반드시 의료 전문가와 상담해 진행하세요.

## 1. 초기 설정

필요한 라이브러리를 임포트하고 환경 변수를 불러온 뒤 `AIProjectClient`를 초기화합니다.

먼저 아래 설치 셀을 1회 실행한 다음 진행하세요.

필수 환경 변수:
- `PROJECT_ENDPOINT`
- `PROJECT_API_KEY`
- `MODEL_NAME`

In [ ]:
# 이 노트북에서 필요한 패키지 설치 (커널당 1회)
%pip install -q python-dotenv azure-ai-projects azure-ai-inference azure-identity azure-core azure-ai-evaluation pandas

In [ ]:
import os
import time
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential
from azure.ai.projects import AIProjectClient

def _clean_env(value: str | None) -> str | None:
    if value is None:
        return None
    return value.strip().strip('"').strip("'")

# .env 탐색: 현재 폴더/상위 폴더 모두 확인
dotenv_candidates = [
    Path.cwd() / '.env',
    Path.cwd().parent / '.env',
    Path('.env'),
    Path('..') / '.env',
]
dotenv_path = next((p for p in dotenv_candidates if p.exists()), None)
if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"✅ .env 로드 완료(덮어쓰기): {dotenv_path}")
else:
    print("⚠️ .env 파일을 찾지 못했습니다. 현재/상위 폴더를 확인하세요.")

project_endpoint = _clean_env(os.environ.get("PROJECT_ENDPOINT"))
project_api_key = _clean_env(os.environ.get("PROJECT_API_KEY"))
endpoint_source = "PROJECT_ENDPOINT"

project_client = None
if not project_endpoint:
    print("❌ 필수 환경 변수 누락: PROJECT_ENDPOINT")
else:
    print(f"ℹ️ 사용 엔드포인트 변수: {endpoint_source}")
    print(f"ℹ️ 사용 엔드포인트 값: {project_endpoint}")

    parsed = urlparse(project_endpoint)
    path = parsed.path or ""
    print(f"ℹ️ 엔드포인트 host: {parsed.netloc}")
    print(f"ℹ️ 엔드포인트 path: {path}")

    if "/api/projects/" in path:
        project_name = path.split("/api/projects/")[-1]
        print(f"✅ 프로젝트 엔드포인트 형식으로 보입니다. (project: {project_name})")
    elif "/models" in path:
        print("⚠️ 모델 엔드포인트 형식입니다. Agent Service에는 부적합할 수 있습니다.")
    else:
        print("⚠️ 엔드포인트 형식을 다시 확인하세요. (/api/projects/<name> 권장)")

    # 기본은 Entra ID(DefaultAzureCredential), 실패 시 PROJECT_API_KEY로 폴백
    try:
        project_client = AIProjectClient(
            endpoint=project_endpoint,
            credential=DefaultAzureCredential(),
        )
        print("✅ AIProjectClient 생성 완료 (DefaultAzureCredential)")
    except Exception as e1:
        if project_api_key:
            try:
                project_client = AIProjectClient(
                    endpoint=project_endpoint,
                    credential=AzureKeyCredential(project_api_key),
                )
                print("✅ AIProjectClient 생성 완료 (PROJECT_API_KEY)")
            except Exception as e2:
                print("❌ AIProjectClient 생성 실패 (DefaultAzureCredential/PROJECT_API_KEY)")
                print("   - DefaultAzureCredential 오류:", e1)
                print("   - PROJECT_API_KEY 오류:", e2)
        else:
            print("❌ AIProjectClient 생성 실패: PROJECT_API_KEY가 없어 폴백 불가")
            print("   - DefaultAzureCredential 오류:", e1)

# 추가 디버깅: 인증 토큰/Agent Service 접근 테스트
try:
    cred = DefaultAzureCredential()
    token = cred.get_token("https://management.azure.com/.default")
    print(f"✅ DefaultAzureCredential 토큰 획득 성공 (만료시각: {token.expires_on})")
except Exception as e:
    print("❌ DefaultAzureCredential 토큰 획득 실패:", e)

if project_client is None:
    print("❌ project_client가 None 입니다. 초기화 셀을 다시 확인하세요.")
else:
    try:
        list_fn = getattr(project_client.agents, "list_agents", None)
        if callable(list_fn):
            _ = list(list_fn())[:1]
            print("✅ agents.list_agents 호출 성공")
        else:
            print("ℹ️ agents.list_agents 메서드를 찾지 못했습니다.")
    except Exception as e:
        print("❌ agents.list_agents 호출 실패:", e)

    try:
        probe_thread = project_client.agents.threads.create()
        print(f"✅ threads.create 호출 성공 (probe thread id: {probe_thread.id})")
    except Exception as e:
        print("❌ threads.create 호출 실패:", e)

## 2. Fun & Fit 건강 어드바이저 에이전트 생성

전반적인 건강/웰니스 안내를 제공하는 에이전트를 생성합니다.

에이전트 지시문에 의료 안내 문구를 포함해, 과도한 확정 표현을 피하고 전문가 상담을 유도하도록 설정합니다.

In [ ]:
def create_health_advisor_agent():
    """의료 안내 문구를 포함한 건강 어드바이저 에이전트를 생성합니다."""
    model_name = os.environ.get("MODEL_NAME")

    if project_client is None:
        print("❌ project_client가 초기화되지 않았습니다.")
        return None
    if not model_name:
        print("❌ 필수 환경 변수 누락: MODEL_NAME")
        return None

    try:
        agent = project_client.agents.create_agent(
            model=model_name,
            name="fun-fit-health-advisor",
            instructions="""
            당신은 친절한 한국어 건강/피트니스 어드바이저입니다.
            항상 다음 원칙을 지키세요:
            1) 의료 진단/치료를 단정하지 말 것
            2) 답변에 의료 안내 문구를 포함할 것
            3) 필요 시 전문 의료진 상담을 권장할 것
            4) 운동/영양은 안전하고 균형 잡힌 접근을 제시할 것
            """,
        )
        print(f"🎉 에이전트 생성 완료: {agent.name} (ID: {agent.id})")
        return agent
    except Exception as e:
        print("❌ 에이전트 생성 오류:", e)
        print(f"- endpoint_source: {endpoint_source}")
        print(f"- project_endpoint: {project_endpoint}")
        print(f"- MODEL_NAME: {model_name}")

        status = getattr(e, "status_code", None)
        if status is not None:
            print(f"- status_code: {status}")

        resp = getattr(e, "response", None)
        if resp is not None:
            try:
                print(f"- response status: {resp.status_code}")
                print(f"- response reason: {getattr(resp, 'reason', 'N/A')}")
            except Exception:
                pass
        return None

health_advisor = create_health_advisor_agent()

## 3. 건강 대화 스레드 만들기

스레드는 사용자 메시지와 에이전트 응답이 누적되는 대화 컨테이너입니다.

건강/피트니스 Q&A 전용 스레드를 생성합니다.

In [ ]:
# Function to create a new conversation thread for health discussions
def start_health_conversation():
    """Create a new thread for health & fitness discussions."""
    try:
        # Create a thread for communication
        thread = project_client.agents.threads.create()
        print(f"Created thread, ID: {thread.id}")
        
        # Return the created thread object so we can use it later
        return thread
    except Exception as e:
        # If thread creation fails, print error and return None
        print(f"❌ Error creating thread: {str(e)}")
        return None

# Initialize a new conversation thread that we'll use for our health Q&A
health_thread = start_health_conversation()

## 4. 건강/피트니스 질문 실행

사용자 질문을 스레드에 추가하고, 에이전트 실행(run)을 통해 답변을 생성합니다.

예시로 BMI 해석과 운동 병행 식단 관련 질문을 사용합니다.

In [ ]:
def ask_health_question(thread_id, user_question):
    """사용자 질문을 스레드에 추가합니다."""
    try:
        return project_client.agents.messages.create(
            thread_id=thread_id,
            role="user",
            content=user_question,
        )
    except Exception as e:
        print("❌ 사용자 메시지 추가 오류:", e)
        return None

def process_thread_run(thread_id, agent_id):
    """에이전트 실행(run)을 생성하고 완료될 때까지 대기합니다."""
    try:
        run = project_client.agents.runs.create_and_process(
            thread_id=thread_id,
            agent_id=agent_id,
        )

        while run.status in ["queued", "in_progress", "requires_action"]:
            time.sleep(1)
            run = project_client.agents.get_run(thread_id=thread_id, run_id=run.id)

        print(f"🤖 실행 완료 상태: {run.status}")
        return run
    except Exception as e:
        print("❌ 실행 처리 오류:", e)
        return None

def _extract_message_text(message):
    """SDK 메시지 객체에서 사람이 읽기 쉬운 텍스트를 추출합니다."""
    texts = []
    for part in getattr(message, "content", []) or []:
        text_obj = getattr(part, "text", None)
        value = getattr(text_obj, "value", None) if text_obj else None
        if value:
            texts.append(value)
    return "\n".join(texts) if texts else str(getattr(message, "content", ""))

def view_thread_messages(thread_id):
    """스레드 메시지를 출력합니다."""
    try:
        messages = project_client.agents.messages.list(thread_id=thread_id)
        for message in messages:
            content_text = _extract_message_text(message)
            print(f"[{message.role}] {content_text}\n")
    except Exception as e:
        print("❌ 스레드 조회 오류:", e)

### 예시 질문 실행

에이전트가 의료 안내 문구를 포함해 응답하는지 확인하기 위해 예시 질문 2개를 실행합니다.
- BMI 계산/해석
- 주 3회 운동 기준 균형 식단

In [ ]:
if health_advisor and health_thread:
    msg1 = ask_health_question(
        health_thread.id,
        "BMI는 어떻게 계산하나요? 그리고 결과는 어떻게 해석하면 좋을까요?",
    )
    run1 = process_thread_run(health_thread.id, health_advisor.id)

    msg2 = ask_health_question(
        health_thread.id,
        "주 3회 운동하는 사람을 위한 균형 잡힌 식단 예시를 알려주세요.",
    )
    run2 = process_thread_run(health_thread.id, health_advisor.id)

    view_thread_messages(health_thread.id)
else:
    print("❌ 예시 질의를 실행할 수 없습니다. agent/thread 초기화를 확인하세요.")

## 5. Evaluation 실행

이 셀에서는 `RougeScoreEvaluator`를 사용해 평가를 진행합니다.
- `RougeScoreEvaluator`: 생성 답변(`response`)과 기준 답변(`ground_truth`)의 유사도 측정

단일 기준답안만 쓰면 표현 차이 때문에 점수가 과하게 낮아질 수 있어,
질문마다 기준답안을 여러 개 두고 `최대(max) rouge_1_f1`를 사용합니다.

### 결과 컬럼 설명
- `rouge_1_precision`: 모델 응답 단어 중 기준답안과 겹치는 비율
- `rouge_1_recall`: 기준답안 단어 중 모델 응답에서 찾아낸 비율
- `rouge_1_f1`: `precision`과 `recall`의 조화 평균(균형 점수)

### 점수 범위 해석 가이드
- 공통 범위: `0.0 ~ 1.0` (클수록 유사도 높음)
- `0.00 ~ 0.10`: 매우 낮음 (단어 겹침이 거의 없음)
- `0.10 ~ 0.30`: 낮음 (핵심어 일부만 일치)
- `0.30 ~ 0.50`: 보통 (핵심 주제는 맞지만 표현 차이 큼)
- `0.50 ~ 0.70`: 높음 (핵심 내용이 상당히 유사)
- `0.70 ~ 1.00`: 매우 높음 (표현/구조까지 매우 유사)

### `rouge_1_f1`이란?
- `ROUGE-1`: 단어(unigram) 단위 겹침 정도
- `F1`: 정밀도(precision)와 재현율(recall)의 조화 평균
- 해석: 값이 `1`에 가까울수록 기준 답변과 핵심 단어가 더 잘 맞고, `0`에 가까울수록 겹침이 적습니다.

주의: 표현이 달라도 의미가 같으면 낮게 나올 수 있으므로,
`rouge_1_f1`은 "빠른 비교용" 지표로 해석하는 것이 좋습니다.

In [ ]:
import re
import unicodedata
import pandas as pd
from azure.ai.evaluation import RougeScoreEvaluator

evaluation_cases = [
    {
        "name": "BMI 해석",
        "prompt": "BMI는 어떻게 계산하고, 수치별로 어떤 의미인가요?",
        "ground_truths": [
            "BMI는 체중(kg)을 키(m)의 제곱으로 나눈 체질량지수입니다. 수치 구간(저체중, 정상, 과체중, 비만)을 참고하되 개인 상태에 따라 전문가 상담이 필요할 수 있습니다.",
            "BMI는 몸무게를 키의 제곱으로 나눠 계산하며 체중 상태를 대략 분류하는 지표입니다. 결과 해석은 개인 건강 상태와 함께 보는 것이 좋습니다.",
        ],
    },
    {
        "name": "운동+식단",
        "prompt": "주 3회 운동하는 직장인을 위한 하루 식단 예시를 알려주세요.",
        "ground_truths": [
            "주 3회 운동하는 직장인은 단백질, 복합 탄수화물, 채소, 건강한 지방을 균형 있게 포함한 식단이 좋습니다. 운동 전후 수분 섭취와 회복 식사를 함께 관리하는 것이 중요합니다.",
            "운동하는 직장인 식단은 단백질 섭취, 탄수화물 타이밍, 채소와 수분 보충을 중심으로 구성하면 좋고 과도한 제한식보다 지속 가능한 균형 식단이 중요합니다.",
        ],
    },
    {
        "name": "수면/회복",
        "prompt": "운동 후 회복을 위해 수면과 스트레칭은 어떻게 관리하면 좋을까요?",
        "ground_truths": [
            "운동 후 회복을 위해 충분한 수면 시간을 확보하고 규칙적인 수면 습관을 유지하세요. 운동 후에는 무리하지 않는 스트레칭으로 근육 긴장을 완화하고 회복을 돕는 것이 좋습니다.",
            "회복을 위해 수면 루틴을 일정하게 유지하고 운동 후 가벼운 스트레칭으로 근육 피로를 줄이는 것이 좋습니다. 통증이 지속되면 강도를 낮추고 전문가 상담을 고려하세요.",
            "운동 후 회복에 있어 수면과 스트레칭은 매우 중요한 역할을 합니다. 충분한 수면을 취하고 운동 뒤 가벼운 스트레칭을 통해 근육을 이완하면 회복에 도움이 됩니다.",
            "회복을 높이려면 일정한 수면 루틴과 적절한 스트레칭 루틴을 함께 관리하세요. 과도한 강도보다는 꾸준한 수면 관리와 스트레칭이 피로 회복에 효과적입니다.",
        ],
    },
]


def normalize_for_rouge(text):
    """Korean/English text normalization for more stable ROUGE overlap."""
    normalized = unicodedata.normalize("NFKC", text or "")
    normalized = normalized.lower()
    normalized = re.sub(r"[^0-9a-zA-Z가-힣\s]", " ", normalized)
    normalized = re.sub(r"\s+", " ", normalized).strip()

    # Korean keyword aliasing: add canonical tokens to reduce tokenizer bias on Korean text
    alias_map = {
        "운동": "exercise",
        "회복": "recovery",
        "수면": "sleep",
        "스트레칭": "stretching",
        "근육": "muscle",
        "피로": "fatigue",
        "식단": "diet",
        "단백질": "protein",
        "탄수화물": "carb",
        "채소": "vegetable",
        "체질량": "bmi",
        "몸무게": "weight",
        "체중": "weight",
        "키": "height",
    }

    alias_tokens = [token for kw, token in alias_map.items() if kw in normalized]
    if alias_tokens:
        normalized = normalized + " " + " ".join(alias_tokens)
    return normalized


def collect_agent_responses(agent, cases):
    rows = []

    if not agent:
        print("❌ 평가를 실행할 에이전트가 없습니다.")
        return rows

    for case in cases:
        print(f"\n=== 평가 실행: {case['name']} ===")

        eval_thread = start_health_conversation()
        if not eval_thread:
            continue

        ask_health_question(eval_thread.id, case["prompt"])
        run = process_thread_run(eval_thread.id, agent.id)

        if not run or run.status != "completed":
            print("❌ run이 정상 완료되지 않았습니다.")
            continue

        messages = project_client.agents.messages.list(thread_id=eval_thread.id)

        response_text = ""
        for message in messages:
            if message.role == "assistant":
                response_text = _extract_message_text(message)
                break

        rows.append(
            {
                "name": case["name"],
                "query": case["prompt"],
                "response": response_text,
                "ground_truths": case["ground_truths"],
            }
        )

    return rows


def run_rouge_evaluator(rows):
    rouge_eval = RougeScoreEvaluator(rouge_type="rouge1")
    results = []

    for row in rows:
        response_norm = normalize_for_rouge(row["response"])

        best = {
            "precision": None,
            "recall": None,
            "f1": -1.0,
            "ref_index": None,
            "ref_text": "",
        }

        for idx, ref in enumerate(row["ground_truths"], start=1):
            ref_norm = normalize_for_rouge(ref)
            try:
                rouge_result = rouge_eval(response=response_norm, ground_truth=ref_norm)
                f1 = rouge_result.get("rouge_f1_score")
                if isinstance(f1, (int, float)) and f1 > best["f1"]:
                    best["f1"] = f1
                    best["precision"] = rouge_result.get("rouge_precision")
                    best["recall"] = rouge_result.get("rouge_recall")
                    best["ref_index"] = idx
                    best["ref_text"] = ref
            except Exception as e:
                print(f"⚠️ Rouge 평가 실패 ({row['name']}, ref {idx}): {e}")

        record = {
            "name": row["name"],
            "rouge_1_precision": best["precision"],
            "rouge_1_recall": best["recall"],
            "rouge_1_f1": None if best["f1"] < 0 else best["f1"],
            "matched_ref": best["ref_index"],
            "response_preview": row["response"][:90].replace("\n", " "),
            "ref_preview": best["ref_text"][:90].replace("\n", " "),
        }
        results.append(record)

    return results


evaluation_rows = collect_agent_responses(health_advisor, evaluation_cases)
evaluation_results = run_rouge_evaluator(evaluation_rows)

if evaluation_results:
    df_eval = pd.DataFrame(evaluation_results)
    display(df_eval)

    if df_eval["rouge_1_f1"].notna().any():
        print(f"\n📊 평균 rouge_1_f1 (max over refs): {df_eval['rouge_1_f1'].dropna().mean():.4f}")
    else:
        print("\n⚠️ rouge_1_f1 점수가 생성되지 않았습니다.")
else:
    print("\n평가 결과가 없습니다.")

## 6. 정리하기

실습이 끝나면 생성한 에이전트를 삭제해 리소스를 정리할 수 있습니다.

(필요하면 에이전트를 유지해 후속 실험에 재사용해도 됩니다.)

In [ ]:
def cleanup(agent):
    """실습에서 생성한 에이전트를 삭제합니다."""
    if not agent:
        print("정리할 에이전트가 없습니다.")
        return

    try:
        project_client.agents.delete_agent(agent.id)
        print(f"🗑️ 에이전트 삭제 완료: {agent.name}")
    except Exception as e:
        print("❌ 에이전트 정리 오류:", e)

cleanup(health_advisor)

## 마무리

이 노트북에서 다음 흐름을 완료했습니다.
1. `AIProjectClient` 초기화
2. 건강 어드바이저 에이전트 생성
3. 스레드 생성 및 질의 실행
4. 응답 확인
5. Foundry 평가 라이브러리 기반 `Rouge` 평가 실행
6. 리소스 정리

> 중요) 실습이 끝난 뒤에는 **리소스 그룹을 삭제**해 불필요한 비용이 발생하지 않도록 정리하세요.

Azure Portal에서 리소스 그룹 삭제:
1. [Azure Portal](https://portal.azure.com) 접속
2. 상단 검색창에서 `리소스 그룹` 검색 후 이동
3. 실습에 사용한 리소스 그룹 선택
4. 상단 `리소스 그룹 삭제` 클릭
5. 확인 입력란에 리소스 그룹 이름을 정확히 입력 후 `삭제`